In [2]:
import os

for root, dirs, files in os.walk("/home/sabrina/AI-Badminton-111"):
    if "merged_scores_imu.csv" in files:
        print(os.path.join(root, "merged_scores_imu.csv"))

/home/sabrina/AI-Badminton-111/merged_scores_imu.csv


In [ ]:
import os
import re
import pandas as pd

folders = {
    "挑": "/home/sabrina/AI-Badminton-111/挑cut",
    "推": "/home/sabrina/AI-Badminton-111/推cut"
}

rows = []

for stroke_type, folder in folders.items():
    for f in os.listdir(folder):
        if not f.endswith(".csv"):
            continue

        m = re.match(r"^(\d+)(挑|推)(補)?_(\d+)\.csv$", f)

        if m:
            rows.append({
                "file_name": f,
                "subject_id": m.group(1),
                "stroke_type_x": stroke_type,
                "shot_id": int(m.group(4)),
                "path": os.path.join(folder, f)
            })

cut_df = pd.DataFrame(rows)

score_df = pd.read_csv("merged_scores_imu.csv")

score_cols = [
    "揮拍軌跡正確度",
    "揮拍速度流暢度",
    "手腕轉動時機正確度",
    "擊球時機正確度",
    "擊球位置正確度"
]

score_df["subject_id"] = score_df["file_name"].astype(str).str.extract(r"^(\d+)")

score_small = score_df[
    ["subject_id", "stroke_type_x"] + score_cols
].drop_duplicates()

merged_df = cut_df.merge(
    score_small,
    on=["subject_id", "stroke_type_x"],
    how="left"
)

merged_df = merged_df.dropna(subset=score_cols).reset_index(drop=True)

print("cut_df:", cut_df.shape)
print("merged_df:", merged_df.shape)
print("缺失值:")
print(merged_df[score_cols].isna().sum())
display(merged_df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'merged_scores_imu.csv'

: 

In [2]:
def convert_score_to_category(score):
    score = int(score)
    return (score - 1) // 2   # 1,2 -> 0 ; 3,4 -> 1 ; 5 -> 2

score_cols = [
    "揮拍軌跡正確度",
    "揮拍速度流暢度",
    "手腕轉動時機正確度",
    "擊球時機正確度",
    "擊球位置正確度"
]

for col in score_cols:
    merged_df[col + "_cat"] = merged_df[col].apply(convert_score_to_category)

y1 = merged_df["揮拍軌跡正確度_cat"].values
y2 = merged_df["揮拍速度流暢度_cat"].values
y3 = merged_df["手腕轉動時機正確度_cat"].values
y4 = merged_df["擊球時機正確度_cat"].values
y5 = merged_df["擊球位置正確度_cat"].values

print("y1:", y1.shape)
print("y2:", y2.shape)
print("y3:", y3.shape)
print("y4:", y4.shape)
print("y5:", y5.shape)

NameError: name 'merged_df' is not defined

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

# =============================
# 先跑 model1
# =============================
metric_name = "揮拍軌跡正確度"
model_path = "專題/models/model1.keras"
y_all = y1

ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
repeat_times = 3
epochs = 30

# =============================
# Scratch model
# =============================
def build_scratch_model(input_shape=(125, 6), num_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(16),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# Transfer model
# =============================
def build_transfer_model(base_model, num_classes=3):
    model = tf.keras.models.clone_model(base_model)
    model.set_weights(base_model.get_weights())

    model.pop()

    for layer in model.layers:
        layer.trainable = False

    model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# Load pretrained model
# =============================
base_model_original = tf.keras.models.load_model(model_path)

all_results = []

for stroke_name, idx in [("挑球", 挑_idx), ("平推", 推_idx)]:

    print("正在跑:", metric_name, "-", stroke_name)

    X_target = X[idx]
    y_target = y_all[idx]

    target_df = merged_df[idx].reset_index(drop=True)
    groups = target_df["subject_id"]

    # 固定 train/test：以人為單位切
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    train_idx, test_idx = next(
        splitter.split(X_target, y_target, groups=groups)
    )

    X_test = X_target[test_idx]
    y_test = y_target[test_idx]

    train_subjects = np.array(sorted(groups.iloc[train_idx].unique()))

    print("train subjects:", len(train_subjects))
    print("test subjects:", groups.iloc[test_idx].nunique())
    print("overlap:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

    for run in range(repeat_times):

        seed = 42 + run
        rng = np.random.default_rng(seed)

        # 這裡固定一組受試者順序
        ordered_subjects = rng.permutation(train_subjects)

        for r in ratios:

            n_subjects = max(1, int(len(ordered_subjects) * r))
            selected_subjects = ordered_subjects[:n_subjects]

            small_idx = target_df[
                target_df["subject_id"].isin(selected_subjects)
            ].index.values

            X_small = X_target[small_idx]
            y_small = y_target[small_idx]

            print(
                f"{stroke_name} | run {run+1} | ratio {int(r*100)}% | "
                f"train samples: {len(X_small)} | "
                f"train subjects: {len(selected_subjects)}"
            )

            early_stop = tf.keras.callbacks.EarlyStopping(
                monitor="loss",
                patience=5,
                restore_best_weights=True
            )

            # ---- Scratch ----
            tf.keras.backend.clear_session()

            scratch_model = build_scratch_model(
                input_shape=X_target.shape[1:]
            )

            scratch_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                callbacks=[early_stop]
            )

            pred_scratch = scratch_model.predict(X_test, verbose=0)
            acc_scratch = accuracy_score(
                y_test,
                np.argmax(pred_scratch, axis=1)
            )

            # ---- Transfer ----
            tf.keras.backend.clear_session()

            transfer_model = build_transfer_model(base_model_original)

            transfer_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                callbacks=[early_stop]
            )

            pred_transfer = transfer_model.predict(X_test, verbose=0)
            acc_transfer = accuracy_score(
                y_test,
                np.argmax(pred_transfer, axis=1)
            )

            all_results.append({
                "維度": metric_name,
                "動作": stroke_name,
                "run": run + 1,
                "資料比例": f"{int(r*100)}%",
                "train_subjects": len(selected_subjects),
                "train_samples": len(X_small),
                "Scratch_acc": acc_scratch,
                "Transfer_acc": acc_transfer
            })


raw_result_df = pd.DataFrame(all_results)
display(raw_result_df)

# 取平均與 std
result_df = raw_result_df.groupby(
    ["維度", "動作", "資料比例"],
    as_index=False
).agg(
    Scratch_mean=("Scratch_acc", "mean"),
    Scratch_std=("Scratch_acc", "std"),
    Transfer_mean=("Transfer_acc", "mean"),
    Transfer_std=("Transfer_acc", "std"),
    train_subjects=("train_subjects", "mean"),
    train_samples=("train_samples", "mean")
)

display(result_df)

# 存檔
result_df.to_csv(
    f"{metric_name}_累加受試者_finetune.csv",
    index=False,
    encoding="utf-8-sig"
)

raw_result_df.to_csv(
    f"{metric_name}_累加受試者_finetune_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

I0000 00:00:1779692551.253319    9911 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779692551.265064    9911 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779692552.376233    9911 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779692557.633209    9911 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

NameError: name 'y1' is not defined